In [1]:
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

from torchvision.models import (
    resnet50, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    vgg16, VGG16_Weights,
    vit_b_16, ViT_B_16_Weights,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Device: cuda
GPU: Tesla T4
CUDA version: 12.8


In [2]:
DATASET_ROOT = "/kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion"
IMAGE_DIR = f"{DATASET_ROOT}/images"
PARSING_DIR = f"{DATASET_ROOT}/segm"


FASHION_CLIP_METADATA_CSV = "a"

OUTPUT_DIR = Path("/kaggle/working/data/processed/embeddings")

CLOTHING_CLASSES = [1, 2, 3, 4, 5, 6, 21]
PADDING_RATIO = 0.05

BATCH_SIZE = 32
NUM_WORKERS = 2

# None -> full 12,701 | 10 -> quick test | 100 -> validation test
MAX_SAMPLES = None

print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_DIR  :", OUTPUT_DIR)
print("MAX_SAMPLES :", MAX_SAMPLES)

DATASET_ROOT: /kaggle/input/datasets/khanhngnguynl/deepfashion/deepfashion
OUTPUT_DIR  : /kaggle/working/data/processed/embeddings
MAX_SAMPLES : None


In [3]:
def build_valid_pairs_from_scratch(image_dir, parsing_dir):
    image_files_all = sorted([f.name for f in Path(image_dir).iterdir() if f.is_file()])
    parsing_files_all = sorted([f.name for f in Path(parsing_dir).iterdir() if f.is_file()])
    parsing_stems_set = set(Path(p).stem for p in parsing_files_all)

    pairs = []
    for image_name in image_files_all:
        candidate_stem = Path(image_name).stem + "_segm"
        if candidate_stem in parsing_stems_set:
            pairs.append((Path(image_name).stem, str(Path(image_dir) / image_name),
                           str(Path(parsing_dir) / (candidate_stem + ".png"))))
    return pairs


def load_metadata(fashion_clip_csv, image_dir, parsing_dir):
    csv_path = Path(fashion_clip_csv)
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        records = []
        for _, row in df.iterrows():
            image_id = row["image_id"]
            image_path = row["image_path"]
            parsing_path = str(Path(parsing_dir) / f"{image_id}_segm.png")
            records.append((image_id, image_path, parsing_path))
        print(f"Đã load metadata có sẵn từ: {csv_path} ({len(records)} rows)")
        return records
    else:
        print(f"[WARNING] Không tìm thấy {csv_path}, tự build lại danh sách valid pairs.")
        records = build_valid_pairs_from_scratch(image_dir, parsing_dir)
        print(f"Đã build lại {len(records)} valid pairs.")
        return records


all_records = load_metadata(FASHION_CLIP_METADATA_CSV, IMAGE_DIR, PARSING_DIR)

records = all_records[:MAX_SAMPLES] if MAX_SAMPLES is not None else all_records
print(f"Số lượng samples sẽ chạy: {len(records)} (MAX_SAMPLES = {MAX_SAMPLES})")

[WARNING] Không tìm thấy a, tự build lại danh sách valid pairs.
Đã build lại 12701 valid pairs.
Số lượng samples sẽ chạy: 12701 (MAX_SAMPLES = None)


In [4]:
def compute_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return (xs.min(), ys.min(), xs.max(), ys.max())


def apply_padding(bbox, w, h, ratio):
    min_x, min_y, max_x, max_y = bbox
    pad_x = int((max_x - min_x) * ratio)
    pad_y = int((max_y - min_y) * ratio)
    return (
        max(0, min_x - pad_x),
        max(0, min_y - pad_y),
        min(w - 1, max_x + pad_x),
        min(h - 1, max_y + pad_y),
    )


def load_cropped_image(image_path, parsing_path, clothing_classes, padding_ratio):
    original_pil = Image.open(image_path).convert("RGB")
    original_array = np.array(original_pil)
    parsing_array = np.array(Image.open(parsing_path))

    clothing_mask = np.isin(parsing_array, clothing_classes)
    bbox = compute_bbox(clothing_mask)
    if bbox is None:
        raise ValueError("Empty clothing mask")

    h, w = clothing_mask.shape
    min_x, min_y, max_x, max_y = apply_padding(bbox, w, h, padding_ratio)
    if max_x <= min_x or max_y <= min_y:
        raise ValueError("Invalid bbox after padding")

    cropped_array = original_array[min_y:max_y + 1, min_x:max_x + 1]
    return Image.fromarray(cropped_array)

In [5]:
class CroppedClothingDataset(Dataset):
    def __init__(self, records, clothing_classes, padding_ratio, transform):
        self.records = records
        self.clothing_classes = clothing_classes
        self.padding_ratio = padding_ratio
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        image_id, image_path, parsing_path = self.records[idx]
        try:
            cropped_pil = load_cropped_image(
                image_path, parsing_path, self.clothing_classes, self.padding_ratio
            )
            tensor = self.transform(cropped_pil)
            return tensor, image_id, image_path, None
        except Exception as e:
            return None, image_id, image_path, str(e)


def collate_skip_errors(batch):
    valid_tensors, valid_ids, valid_paths, errors = [], [], [], []
    for tensor, image_id, image_path, err in batch:
        if err is not None or tensor is None:
            errors.append((image_id, image_path, err if err else "Unknown error"))
        else:
            valid_tensors.append(tensor)
            valid_ids.append(image_id)
            valid_paths.append(image_path)

    batch_tensor = torch.stack(valid_tensors) if valid_tensors else None
    return batch_tensor, valid_ids, valid_paths, errors

In [6]:
class FeatureExtractorWrapper(torch.nn.Module):
    """Bọc model gốc, trả về feature TRƯỚC classification head theo đúng architecture."""

    def __init__(self, model, arch_name):
        super().__init__()
        self.arch_name = arch_name

        if arch_name == "resnet50":
            self.backbone = torch.nn.Sequential(*list(model.children())[:-1])
            self.feature_dim = 2048
            self.forward_fn = self._forward_resnet

        elif arch_name == "efficientnet_b0":
            self.features = model.features
            self.avgpool = model.avgpool
            self.feature_dim = 1280
            self.forward_fn = self._forward_efficientnet

        elif arch_name == "vgg16":
            self.features = model.features
            self.avgpool = model.avgpool
            self.classifier = torch.nn.Sequential(*list(model.classifier.children())[:-1])
            self.feature_dim = 4096
            self.forward_fn = self._forward_vgg

        elif arch_name == "vit_b_16":
            self.model = model
            self.model.heads = torch.nn.Identity()
            self.feature_dim = 768
            self.forward_fn = self._forward_vit

        else:
            raise ValueError(f"Unknown architecture: {arch_name}")

    def _forward_resnet(self, x):
        feat = self.backbone(x)
        return torch.flatten(feat, 1)

    def _forward_efficientnet(self, x):
        feat = self.features(x)
        feat = self.avgpool(feat)
        return torch.flatten(feat, 1)

    def _forward_vgg(self, x):
        feat = self.features(x)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        return self.classifier(feat)

    def _forward_vit(self, x):
        return self.model(x)

    def forward(self, x):
        return self.forward_fn(x)


MODEL_REGISTRY = {
    "resnet50": {
        "loader": lambda: resnet50(weights=ResNet50_Weights.DEFAULT),
        "weights": ResNet50_Weights.DEFAULT,
        "expected_dim": 2048,
    },
    "efficientnet_b0": {
        "loader": lambda: efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT),
        "weights": EfficientNet_B0_Weights.DEFAULT,
        "expected_dim": 1280,
    },
    "vgg16": {
        "loader": lambda: vgg16(weights=VGG16_Weights.DEFAULT),
        "weights": VGG16_Weights.DEFAULT,
        "expected_dim": 4096,
    },
    "vit_b_16": {
        "loader": lambda: vit_b_16(weights=ViT_B_16_Weights.DEFAULT),
        "weights": ViT_B_16_Weights.DEFAULT,
        "expected_dim": 768,
    },
}


def load_model(arch_name, device):
    entry = MODEL_REGISTRY[arch_name]
    base_model = entry["loader"]()
    transform = entry["weights"].transforms()

    wrapped_model = FeatureExtractorWrapper(base_model, arch_name)
    wrapped_model.to(device)
    wrapped_model.eval()

    return wrapped_model, transform, entry["expected_dim"]

In [7]:
def extract_embeddings_for_model(arch_name, records, device, batch_size, num_workers):
    print(f"\nLoading model: {arch_name} ...")
    model, transform, expected_dim = load_model(arch_name, device)

    dataset = CroppedClothingDataset(records, CLOTHING_CLASSES, PADDING_RATIO, transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_skip_errors,
    )

    all_embeddings, all_ids, all_paths, failed_samples = [], [], [], []
    start_time = time.time()
    pbar = tqdm(total=len(records), desc=f"Processing {arch_name}")

    with torch.no_grad():
        for batch_tensor, batch_ids, batch_paths, batch_errors in loader:
            for image_id, image_path, err in batch_errors:
                failed_samples.append({"image_id": image_id, "image_path": image_path, "error": err})

            if batch_tensor is not None:
                try:
                    batch_tensor = batch_tensor.to(device)
                    feat = model(batch_tensor)
                    feat = feat.cpu().numpy().astype(np.float32)
                    all_embeddings.append(feat)
                    all_ids.extend(batch_ids)
                    all_paths.extend(batch_paths)
                except torch.cuda.OutOfMemoryError as e:
                    torch.cuda.empty_cache()
                    for image_id, image_path in zip(batch_ids, batch_paths):
                        failed_samples.append({
                            "image_id": image_id, "image_path": image_path,
                            "error": f"CUDA OOM at batch_size={batch_size}: {str(e)}",
                        })
                    print(f"[OOM] Hãy giảm BATCH_SIZE (32→16→8) và chạy lại cell này.")
                except Exception as e:
                    for image_id, image_path in zip(batch_ids, batch_paths):
                        failed_samples.append({"image_id": image_id, "image_path": image_path, "error": str(e)})
                    print(f"[ERROR] Batch lỗi: {e}")
                    traceback.print_exc()

            pbar.update(len(batch_ids) + len(batch_errors))

    pbar.close()
    total_time = time.time() - start_time

    embeddings = np.concatenate(all_embeddings, axis=0) if all_embeddings else np.empty((0, expected_dim), dtype=np.float32)
    return embeddings, all_ids, all_paths, failed_samples, total_time, expected_dim


def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    return embeddings / norms


def validate_embeddings(embeddings, expected_count, arch_name):
    assert embeddings.shape[0] == expected_count, (
        f"[{arch_name}] Số embedding ({embeddings.shape[0]}) không khớp input thành công ({expected_count})"
    )
    has_nan = bool(np.isnan(embeddings).any())
    has_inf = bool(np.isinf(embeddings).any())
    assert not has_nan, f"[{arch_name}] Embedding chứa NaN"
    assert not has_inf, f"[{arch_name}] Embedding chứa Inf"

    norms = np.linalg.norm(embeddings, axis=1)
    return {
        "has_nan": has_nan, "has_inf": has_inf,
        "norm_mean": float(norms.mean()) if len(norms) else float("nan"),
        "norm_min": float(norms.min()) if len(norms) else float("nan"),
        "norm_max": float(norms.max()) if len(norms) else float("nan"),
    }


def save_model_outputs(output_dir, arch_name, embeddings_normalized, image_ids, image_paths, failed_samples):
    model_dir = output_dir / arch_name
    model_dir.mkdir(parents=True, exist_ok=True)

    np.save(model_dir / "cropped_embeddings.npy", embeddings_normalized)
    pd.DataFrame({"image_id": image_ids, "image_path": image_paths}).to_csv(model_dir / "metadata.csv", index=False)

    failed_df = pd.DataFrame(failed_samples) if failed_samples else pd.DataFrame(columns=["image_id", "image_path", "error"])
    failed_df.to_csv(model_dir / "failed_samples.csv", index=False)

    print(f"Saved outputs for {arch_name} -> {model_dir}")


def run_single_model(arch_name, records, device, batch_size, num_workers, output_dir):
    print("\n" + "=" * 40)
    print(arch_name)
    print("=" * 40)

    embeddings, image_ids, image_paths, failed_samples, total_time, expected_dim = \
        extract_embeddings_for_model(arch_name, records, device, batch_size, num_workers)

    embeddings_normalized = normalize_embeddings(embeddings)
    stats = validate_embeddings(embeddings_normalized, len(image_ids), arch_name)
    save_model_outputs(output_dir, arch_name, embeddings_normalized, image_ids, image_paths, failed_samples)

    n_samples = len(image_ids)
    images_per_sec = n_samples / total_time if total_time > 0 else 0

    print(f"\nDevice: {device}")
    print(f"Model: {arch_name}")
    print(f"Number of samples: {n_samples}")
    print(f"\nEmbedding shape: {embeddings_normalized.shape}")
    print(f"Embedding dtype: {embeddings_normalized.dtype}")
    print(f"\nNaN: {stats['has_nan']}")
    print(f"Inf: {stats['has_inf']}")
    print(f"\nNormalized norm:")
    print(f"Mean: {stats['norm_mean']:.6f}")
    print(f"Min : {stats['norm_min']:.6f}")
    print(f"Max : {stats['norm_max']:.6f}")
    print(f"\nSuccessful: {n_samples}")
    print(f"Failed: {len(failed_samples)}")
    print(f"\nProcessing time: {total_time:.2f} s")
    print(f"Images/sec: {images_per_sec:.2f}")

    return {
        "Model": arch_name, "Samples": n_samples, "Dimension": expected_dim,
        "Failed": len(failed_samples), "Time (s)": round(total_time, 2),
        "Images/sec": round(images_per_sec, 2),
    }

In [8]:
result_resnet50 = run_single_model("resnet50", records, device, BATCH_SIZE, NUM_WORKERS, OUTPUT_DIR)


resnet50

Loading model: resnet50 ...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 209MB/s]


Processing resnet50:   0%|          | 0/12701 [00:00<?, ?it/s]

Saved outputs for resnet50 -> /kaggle/working/data/processed/embeddings/resnet50

Device: cuda
Model: resnet50
Number of samples: 12701

Embedding shape: (12701, 2048)
Embedding dtype: float32

NaN: False
Inf: False

Normalized norm:
Mean: 1.000000
Min : 1.000000
Max : 1.000000

Successful: 12701
Failed: 0

Processing time: 185.65 s
Images/sec: 68.41


In [9]:
result_efficientnet_b0 = run_single_model("efficientnet_b0", records, device, BATCH_SIZE, NUM_WORKERS, OUTPUT_DIR)


efficientnet_b0

Loading model: efficientnet_b0 ...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 142MB/s] 


Processing efficientnet_b0:   0%|          | 0/12701 [00:00<?, ?it/s]

Saved outputs for efficientnet_b0 -> /kaggle/working/data/processed/embeddings/efficientnet_b0

Device: cuda
Model: efficientnet_b0
Number of samples: 12701

Embedding shape: (12701, 1280)
Embedding dtype: float32

NaN: False
Inf: False

Normalized norm:
Mean: 1.000000
Min : 1.000000
Max : 1.000000

Successful: 12701
Failed: 0

Processing time: 150.94 s
Images/sec: 84.15


In [10]:
result_vgg16 = run_single_model("vgg16", records, device, BATCH_SIZE, NUM_WORKERS, OUTPUT_DIR)


vgg16

Loading model: vgg16 ...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 213MB/s]


Processing vgg16:   0%|          | 0/12701 [00:00<?, ?it/s]

Saved outputs for vgg16 -> /kaggle/working/data/processed/embeddings/vgg16

Device: cuda
Model: vgg16
Number of samples: 12701

Embedding shape: (12701, 4096)
Embedding dtype: float32

NaN: False
Inf: False

Normalized norm:
Mean: 1.000000
Min : 1.000000
Max : 1.000000

Successful: 12701
Failed: 0

Processing time: 151.71 s
Images/sec: 83.72


In [11]:
result_vit_b16 = run_single_model("vit_b_16", records, device, BATCH_SIZE, NUM_WORKERS, OUTPUT_DIR)


vit_b_16

Loading model: vit_b_16 ...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 208MB/s]


Processing vit_b_16:   0%|          | 0/12701 [00:00<?, ?it/s]

Saved outputs for vit_b_16 -> /kaggle/working/data/processed/embeddings/vit_b_16

Device: cuda
Model: vit_b_16
Number of samples: 12701

Embedding shape: (12701, 768)
Embedding dtype: float32

NaN: False
Inf: False

Normalized norm:
Mean: 1.000000
Min : 1.000000
Max : 1.000000

Successful: 12701
Failed: 0

Processing time: 166.97 s
Images/sec: 76.07


In [12]:
# Chỉ đưa vào summary những model bạn đã thực sự chạy ở các cell trên
results = []
for var_name in ["result_resnet50", "result_efficientnet_b0", "result_vgg16", "result_vit_b16"]:
    if var_name in globals():
        results.append(globals()[var_name])

summary_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)
print(summary_df.to_markdown(index=False))


BENCHMARK SUMMARY
| Model           |   Samples |   Dimension |   Failed |   Time (s) |   Images/sec |
|:----------------|----------:|------------:|---------:|-----------:|-------------:|
| resnet50        |     12701 |        2048 |        0 |     185.65 |        68.41 |
| efficientnet_b0 |     12701 |        1280 |        0 |     150.94 |        84.15 |
| vgg16           |     12701 |        4096 |        0 |     151.71 |        83.72 |
| vit_b_16        |     12701 |         768 |        0 |     166.97 |        76.07 |
